In [1]:
import kagglehub
path = kagglehub.dataset_download("laveshjadon/ai-impact-on-students")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ai-impact-on-students' dataset.
Path to dataset files: /kaggle/input/ai-impact-on-students


In [2]:
import pandas as pd

In [11]:
csv_path = path + '/ai_student_impact_dataset (1).csv'
df = pd.read_csv(csv_path)

df.info()
print('Информация о датасете:')
print(f'Количество строк:{df.shape[0]}')
print(f'Количество столбцов:{df.shape[1]}')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student_ID                  50000 non-null  int64  
 1   Major_Category              50000 non-null  object 
 2   Year_of_Study               50000 non-null  object 
 3   Pre_Semester_GPA            50000 non-null  float64
 4   Weekly_GenAI_Hours          50000 non-null  float64
 5   Primary_Use_Case            50000 non-null  object 
 6   Prompt_Engineering_Skill    50000 non-null  object 
 7   Tool_Diversity              50000 non-null  int64  
 8   Paid_Subscription           50000 non-null  bool   
 9   Traditional_Study_Hours     50000 non-null  float64
 10  Perceived_AI_Dependency     50000 non-null  int64  
 11  Institutional_Policy        50000 non-null  object 
 12  Anxiety_Level_During_Exams  50000 non-null  int64  
 13  Post_Semester_GPA           500

In [4]:
from google.colab import auth

auth.authenticate_user()

In [5]:
from google.cloud import bigquery

project_id = "my-project-savkinam"

client = bigquery.Client(project=project_id)

In [12]:
dataset_id = "colab_klass"
table_name = "ai_student_impact"

table_id = f"{project_id}.{dataset_id}.{table_name}"

#Загрузка в BQ (при повторном запуске не добавляем)
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

job = client.load_table_from_dataframe(
    df,
    table_id,
    job_config=job_config
)

job.result()

print("Таблица успешно загружена")

Таблица успешно загружена


In [15]:
#Проверка количества строк
query = f"""
SELECT COUNT(*) AS total_rows
FROM `{table_id}`
"""

client.query(query).to_dataframe()

,total_rows
0,50000


In [17]:
#Пропуски
query = f"""
SELECT
    COUNTIF(Major_Category IS NULL) AS major_null,
    COUNTIF(Pre_Semester_GPA IS NULL) AS gpa_null,
    COUNTIF(Post_Semester_GPA IS NULL) AS post_gpa_null,
    COUNTIF(Burnout_Risk_Level IS NULL) AS burnout_null
FROM `{table_id}`
"""

client.query(query).to_dataframe()

,major_null,gpa_null,post_gpa_null,burnout_null
0,0,0,0,0


In [20]:
#Распределение по специальностям
print('Распределение по специальностям')
query = f"""
SELECT
Major_Category,
COUNT(*) AS students
FROM `{table_id}`
GROUP BY Major_Category
ORDER BY students DESC;
"""
client.query(query).to_dataframe()

Распределение по специальностям


,Major_Category,students
0,STEM,15059
1,Business,12538
2,Humanities,9994
3,Medical,6476
4,Arts,5933


In [21]:
#Среднее использование ИИ
print('Среднее использование ИИ')
query = f"""
SELECT
AVG(Weekly_GenAI_Hours) AS avg_hours,
MIN(Weekly_GenAI_Hours) AS min_hours,
MAX(Weekly_GenAI_Hours) AS max_hours
FROM `{table_id}`
"""
client.query(query).to_dataframe()

Среднее использование ИИ


,avg_hours,min_hours,max_hours
0,8.427752,0.0,40.0


#Классификация

In [26]:
#Целевая переменная - риск выгорания
query = f"""
CREATE OR REPLACE MODEL `my-project-savkinam.colab_klass.ai_classification_model_colab`
OPTIONS(
    model_type='logistic_reg',
    input_label_cols=['Burnout_Risk_Level']
) AS

SELECT
    Weekly_GenAI_Hours,
    Anxiety_Level_During_Exams,
    Traditional_Study_Hours,
    Pre_Semester_GPA,
    Post_Semester_GPA,
    Burnout_Risk_Level
FROM `{table_id}`
"""
client.query(query).to_dataframe()

""


In [29]:
query = f"""
SELECT *
FROM ML.PREDICT(
    MODEL `my-project-savkinam.colab_klass.ai_classification_model_colab`,
    (
        SELECT
            Weekly_GenAI_Hours,
            Anxiety_Level_During_Exams,
            Traditional_Study_Hours,
            Pre_Semester_GPA,
            Post_Semester_GPA
        FROM `{table_id}`
    )
)
"""
client.query(query).to_dataframe()

,predicted_Burnout_Risk_Level,predicted_Burnout_Risk_Level_probs,Weekly_GenAI_Hours,Anxiety_Level_During_Exams,Traditional_Study_Hours,Pre_Semester_GPA,Post_Semester_GPA
0,Medium,"[{'label': 'Medium', 'prob': 0.486073882022513...",6.79,1,12.03,2.400,2.623
1,Medium,"[{'label': 'Medium', 'prob': 0.489489450470636...",4.54,1,1.05,2.191,2.282
2,Low,"[{'label': 'Low', 'prob': 0.45528794729692806}...",3.38,1,18.11,2.635,2.687
3,High,"[{'label': 'High', 'prob': 0.5518112977103883}...",18.32,1,1.00,2.640,2.963
4,Low,"[{'label': 'Low', 'prob': 0.4590563479380901},...",0.70,1,11.09,2.416,2.589
...,...,...,...,...,...,...,...
49995,Medium,"[{'label': 'Medium', 'prob': 0.468418785244995...",4.15,10,7.08,3.214,3.248
49996,High,"[{'label': 'High', 'prob': 0.6482538629614255}...",20.25,10,6.67,3.250,3.306
49997,High,"[{'label': 'High', 'prob': 0.8548101783224255}...",24.30,10,2.02,1.793,1.506
49998,High,"[{'label': 'High', 'prob': 0.5289787490363471}...",18.60,10,12.26,3.606,3.739


In [30]:
query = f"""
SELECT *
FROM ML.EVALUATE(MODEL `my-project-savkinam.colab_klass.ai_classification_model_colab`);
"""
client.query(query).to_dataframe()

,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,0.546978,0.495073,0.511084,0.508316,0.969338,0.68878
